# 01 · Data acquisition

**Project:** New England natural gas demand & Henry Hub price forecasting
**Author:** Cathy Kam

All inputs are public:

| Data | Source | Frequency | Used for |
|---|---|---|---|
| Massachusetts deliveries to consumers, by end use (MMcf) | EIA natural gas monthly (`N3060MA2`, `N3010MA2`, `N3020MA2`, `N3035MA2`, `N3045MA2`) | Monthly | Demand target |
| Henry Hub spot price ($/MMBtu) | EIA (`RNGWHHD`) | Daily | Price target |
| Working gas in storage, Lower 48 and East (Bcf) | EIA Weekly Natural Gas Storage Report | Weekly | Price driver |
| U.S. LNG exports (MMcf) | EIA (`N9133US2`) | Monthly | Price driver |
| Daily max/min temperature | Open-Meteo ERA5 archive | Daily | HDD/CDD for both models |

EIA series are pulled through the API v2 `seriesid` route. Get a free key at https://www.eia.gov/opendata/register.php and set it as an environment variable before launching Jupyter:

```bash
export EIA_API_KEY=your_key_here      # Windows PowerShell: $env:EIA_API_KEY="your_key_here"
```

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src import config, data, features, plotting
plotting.use_style()
pd.set_option("display.float_format", "{:,.2f}".format)

## Weather → degree days

Gas demand responds to *heating degree days* (HDD = max(65°F − daily mean temp, 0)). Massachusetts weather is a weighted average of Boston, Worcester and Springfield, weighted toward eastern MA where most gas load sits. The Henry Hub model uses a separate national proxy (New York, Chicago, Houston, Atlanta, Philadelphia).

In [ ]:
pd.DataFrame(config.MA_WEATHER_STATIONS).T

## Pull everything
Skip this cell if `data/raw/` is already populated.

In [ ]:
import os
if os.environ.get("EIA_API_KEY"):
    data.pull_all()
else:
    print("EIA_API_KEY not set - loading existing files from data/raw/")

In [ ]:
raw = data.load_raw()
summary = []
for name, obj in raw.items():
    s = obj.iloc[:, 0] if isinstance(obj, pd.DataFrame) else obj
    summary.append({"dataset": name, "rows": len(obj), "start": s.index.min().date(),
                    "end": s.index.max().date(), "missing": int(s.isna().sum())})
pd.DataFrame(summary).set_index("dataset")

## Quick integrity checks

In [ ]:
cons = raw["ma_consumption"]
# End uses should add up to total deliveries (EIA total also includes vehicle fuel, so a small gap is expected)
gap = (cons["total"] - cons[["residential", "commercial", "industrial", "electric_power"]].sum(axis=1)) / cons["total"] * 100
print(f"Sector sum vs total: median gap {gap.median():.2f}%, max {gap.abs().max():.2f}%")

hh = raw["henry_hub"]
print(f"Henry Hub: {hh.index.min().date()} to {hh.index.max().date()}, "
      f"min ${hh.min():.2f}, max ${hh.max():.2f}")
print("Largest daily Henry Hub prices (check these against known events, e.g. Winter Storm Uri, Feb 2021):")
hh.nlargest(5)